# Refresh pipeline runner

This notebook runs the attendance refresh, feeds the refreshed GoHeels CSV into the performance scraper, runs the performance refresh, then executes the midseason scoring notebook so the three app-facing midseason CSVs are overwritten in `project/data/`.

In [ ]:
from pathlib import Path
import shutil
import time
import nbformat
import pandas as pd
from nbconvert.preprocessors import ExecutePreprocessor

NB_DIR = Path.cwd().resolve()         # .../SCRAPERS
SCRAPERS_ROOT = NB_DIR
PROJECT_ROOT = SCRAPERS_ROOT.parent

ATTENDANCE_ROOT = SCRAPERS_ROOT / "GoHeels_Attendance_Scraper"
PERFORMANCE_ROOT = SCRAPERS_ROOT / "Game_Stat_Scraper"
SCORING_ROOT = PROJECT_ROOT / "SCORING"
APP_ROOT = PROJECT_ROOT / "APP"

ATTENDANCE_NOTEBOOK = ATTENDANCE_ROOT / "unc_baseball_scraper_final_0331_incremental_cache_project_paths_ROOTALIGNED.ipynb"
PERFORMANCE_NOTEBOOK = PERFORMANCE_ROOT / "notebooks" / "final_game_stat_scraper_updated_combined_with_performance_features_refresh_cache_project_paths_ROOTALIGNED.ipynb"
SCORING_NOTEBOOK = SCORING_ROOT / "notebooks" / "unified_final_midseason_scoring_PROJECT_ROOT_ALIGNED.ipynb"

ATTENDANCE_FINAL_CSV = ATTENDANCE_ROOT / "unc_baseball_final.csv"

PERFORMANCE_DATA_DIR = PERFORMANCE_ROOT / "data"
PERFORMANCE_EXPORTS_DIR = PERFORMANCE_ROOT / "exports"
PERFORMANCE_DATA_DIR.mkdir(parents=True, exist_ok=True)
PERFORMANCE_EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

PERFORMANCE_GOHEELS_INPUT = PERFORMANCE_DATA_DIR / "unc_baseball_final.csv"
PERFORMANCE_OUTPUT_CSV = PERFORMANCE_EXPORTS_DIR / "uncbaseball_season_performance_0331.csv"

APP_DATA_DIR = APP_ROOT / "data"
APP_DATA_DIR.mkdir(parents=True, exist_ok=True)

REFRESH_RUNS_DIR = SCRAPERS_ROOT / "refresh_runs"
REFRESH_RUNS_DIR.mkdir(parents=True, exist_ok=True)

MIDSEASON_APP_FILES = [
    APP_DATA_DIR / "db_predictions_2024_2026_full_df.csv",
    APP_DATA_DIR / "db_predictions_2024_2026_tier3_raw_tier_cols.csv",
    APP_DATA_DIR / "db_predictions_2024_2026_tier3_final_feats_export.csv",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SCRAPERS_ROOT:", SCRAPERS_ROOT)
print("SCORING_ROOT:", SCORING_ROOT)
print("APP_ROOT:", APP_ROOT)
print("APP_DATA_DIR:", APP_DATA_DIR)
print("ATTENDANCE_NOTEBOOK:", ATTENDANCE_NOTEBOOK)
print("PERFORMANCE_NOTEBOOK:", PERFORMANCE_NOTEBOOK)
print("SCORING_NOTEBOOK:", SCORING_NOTEBOOK)
print("ATTENDANCE_FINAL_CSV:", ATTENDANCE_FINAL_CSV)
print("PERFORMANCE_GOHEELS_INPUT:", PERFORMANCE_GOHEELS_INPUT)
print("PERFORMANCE_OUTPUT_CSV:", PERFORMANCE_OUTPUT_CSV)
print("REFRESH_RUNS_DIR:", REFRESH_RUNS_DIR)

In [ ]:
def execute_notebook(input_path: Path, output_path: Path, cwd: Path, timeout: int = 3600):
    with input_path.open("r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    ep = ExecutePreprocessor(timeout=timeout, kernel_name="python3")
    ep.preprocess(nb, resources={"metadata": {"path": str(cwd)}})

    with output_path.open("w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    return output_path

In [ ]:
run_stamp = time.strftime("%Y%m%d_%H%M%S")

In [ ]:
# 1) Run the attendance scraper notebook
attendance_executed = REFRESH_RUNS_DIR / f"attendance_refresh_{run_stamp}.ipynb"
execute_notebook(
    input_path=ATTENDANCE_NOTEBOOK,
    output_path=attendance_executed,
    cwd=ATTENDANCE_NOTEBOOK.parent,
    timeout=3600,
)

if not ATTENDANCE_FINAL_CSV.exists():
    raise FileNotFoundError(f"Expected attendance output not found: {ATTENDANCE_FINAL_CSV}")

print("Attendance refresh complete:", ATTENDANCE_FINAL_CSV)

In [ ]:
# 2) Feed the refreshed GoHeels CSV directly into the performance scraper
shutil.copy2(ATTENDANCE_FINAL_CSV, PERFORMANCE_GOHEELS_INPUT)
print("Copied refreshed GoHeels CSV ->", PERFORMANCE_GOHEELS_INPUT)

In [ ]:
# 3) Run the performance scraper notebook
performance_executed = REFRESH_RUNS_DIR / f"performance_refresh_{run_stamp}.ipynb"
execute_notebook(
    input_path=PERFORMANCE_NOTEBOOK,
    output_path=performance_executed,
    cwd=PERFORMANCE_NOTEBOOK.parent,
    timeout=7200,
)

if not PERFORMANCE_OUTPUT_CSV.exists():
    raise FileNotFoundError(f"Expected performance output not found: {PERFORMANCE_OUTPUT_CSV}")

print("Performance refresh complete:", PERFORMANCE_OUTPUT_CSV)

In [ ]:
# 4) Run the midseason scoring notebook
scoring_executed = REFRESH_RUNS_DIR / f"midseason_scoring_refresh_{run_stamp}.ipynb"
execute_notebook(
    input_path=SCORING_NOTEBOOK,
    output_path=scoring_executed,
    cwd=SCORING_NOTEBOOK.parent,
    timeout=7200,
)

missing = [str(p) for p in MIDSEASON_APP_FILES if not p.exists()]
if missing:
    raise FileNotFoundError("Expected app midseason outputs not found: " + ", ".join(missing))

print("Midseason scoring refresh complete.")
for p in MIDSEASON_APP_FILES:
    print("  wrote:", p)

In [ ]:
# 5) Quick validation preview
for p in MIDSEASON_APP_FILES:
    df_preview = pd.read_csv(p)
    print(f"{p.name}: {df_preview.shape}")
    if 'season_year' in df_preview.columns:
        print(df_preview['season_year'].value_counts(dropna=False).sort_index())
    print("-" * 80)

pd.read_csv(MIDSEASON_APP_FILES[0]).tail(10)